# Error handling

The library has no exception hierarchy for failed lookups: a failing source does not
make `search_concepts` raise. Failures show up in these places instead:

| Situation | Where it shows up |
| --- | --- |
| API key or optional extra missing | the source is absent from `lookup.adapters` |
| HTTP error inside an adapter | logged; the adapter returns no concepts |
| Source exceeds `timeout_per_source` in a multi-source search | `result.errors` and `result.sources_failed` |
| Concept not found, or the details request failed | `get_concept_details` returns `None` |
| Source keeps failing (health tracking on) | its circuit breaker opens and requests are skipped |
| Invalid configuration | `pydantic.ValidationError` when building `LookupConfig` |
| `add_source` for an unavailable source | `RuntimeError` |

In [ ]:
import asyncio
import logging

from pydantic import ValidationError

from knowledge_lookup import (
    CentralKnowledgeLookup,
    KnowledgeSource,
    LookupConfig,
    create_knowledge_lookup,
)

## Check availability first

Sources that could not be initialised are silently left out of a search, so check
which ones you actually have.

In [ ]:
wanted = [KnowledgeSource.HPO, KnowledgeSource.MONDO, KnowledgeSource.OMIM]
lookup = create_knowledge_lookup(enabled_sources=wanted)
print("Available:", [source.value for source in lookup.get_available_sources()])
print("Not available:", [source.value for source in wanted if source not in lookup.adapters])

result = await lookup.search_concepts("asthma", sources=wanted, max_results=10)
print("Queried:", list(result.sources_queried))
print("Succeeded:", list(result.sources_succeeded))
await lookup.close()

## Errors recorded per source

A deliberately short timeout makes both sources fail. The result still comes back, with
one message per failed source.

In [ ]:
config = LookupConfig(
    enabled_sources=[KnowledgeSource.HPO, KnowledgeSource.MONDO],
    timeout_per_source=0.5,  # unrealistically short
)
lookup = CentralKnowledgeLookup(config)
result = await lookup.search_concepts("asthma")
print("Concepts:", result.total_found)
print("Failed:", list(result.sources_failed))
for source, message in result.errors.items():
    print(f"  {source}: {message}")
await lookup.close()

## Retry, then fall back

Because failures come back as empty results or `result.errors`, retry logic checks the
result rather than catching exceptions. DrugBank returns nothing here, so the search
falls back to PubChem.

In [ ]:
async def search_with_fallback(query, preferred, fallback, attempts=2):
    lookup = create_knowledge_lookup(enabled_sources=preferred + fallback)
    try:
        for attempt in range(1, attempts + 1):
            result = await lookup.search_concepts(query, sources=preferred, max_results=5)
            if result.concepts:
                return result
            print(f"attempt {attempt}: no concepts, errors={result.errors}")
            await asyncio.sleep(attempt)  # back off a little before retrying
        print("falling back to", [source.value for source in fallback])
        return await lookup.search_concepts(query, sources=fallback, max_results=5)
    finally:
        await lookup.close()


result = await search_with_fallback(
    "aspirin", preferred=[KnowledgeSource.DRUGBANK], fallback=[KnowledgeSource.PUBCHEM]
)
print([(concept.primary_label, concept.primary_id) for concept in result.concepts])

## Source health and circuit breakers

With `enable_source_health_tracking=True`, each source gets a circuit breaker. After
`circuit_breaker_threshold` consecutive failures (default 5) the breaker opens, and for
`circuit_breaker_cooldown` seconds (default 30) the adapter's HTTP requests are refused
with `CircuitBreakerOpen` instead of being sent. Every search result carries a health
snapshot in `result.source_health`.

In [ ]:
config = LookupConfig(
    enabled_sources=[KnowledgeSource.HPO],
    enable_source_health_tracking=True,
    circuit_breaker_threshold=3,
    circuit_breaker_cooldown=60.0,
)
lookup = CentralKnowledgeLookup(config)

result = await lookup.search_concepts("seizure", max_results=3)
print("Concepts:", result.total_found)
for health in result.source_health.values():
    print(
        f"  {health.source}: {health.circuit_state}, health {health.health_score}, calls {health.total_calls}"
    )

# Simulate three failed calls to open the breaker.
breaker = lookup.health_tracker.get_or_create(KnowledgeSource.HPO)
for _ in range(config.circuit_breaker_threshold):
    breaker.record_failure()

result = await lookup.search_concepts("seizure", max_results=3)
print("Concepts while the breaker is open:", result.total_found)
health = lookup.health_tracker.get_health(KnowledgeSource.HPO)
print(f"  HPO: {health.circuit_state}, {health.failure_count} consecutive failures")
await lookup.close()

## See adapter errors in the log

HTTP errors that an adapter handles itself are only visible in the log, under the
`knowledge_lookup` logger. Here the COSMIC endpoint answers 404: the search returns no
concepts and no `result.errors`, but the log says why.

**Warning:** log messages can contain request URLs, and BioPortal sends its API key as a
URL parameter. Check logs for keys before sharing them.

In [ ]:
logging.basicConfig(level=logging.ERROR, format="%(levelname)s %(name)s: %(message)s", force=True)

lookup = create_knowledge_lookup(enabled_sources=[KnowledgeSource.COSMIC])
result = await lookup.search_concepts("BRAF", sources=[KnowledgeSource.COSMIC])
print("Concepts:", result.total_found, "errors:", result.errors)
await lookup.close()

## Errors that do raise

Invalid configuration and explicit requests for an unavailable source raise.

In [ ]:
try:
    LookupConfig(cache_enabled=True)  # not a LookupConfig field
except ValidationError as exc:
    print("Invalid configuration:", exc.errors()[0]["msg"])

lookup = CentralKnowledgeLookup(LookupConfig(enabled_sources=[KnowledgeSource.HPO]))
try:
    await lookup.add_source(KnowledgeSource.OMIM)
    print("OMIM added")
except RuntimeError as exc:
    print("Cannot add source:", exc)
await lookup.close()

## Summary

- Check `lookup.adapters` before relying on a source.
- Inspect `result.errors`, `result.sources_failed` and empty results instead of catching
  exceptions around `search_concepts`.
- Treat `None` from `get_concept_details` as "not found or unavailable".
- Turn on source health tracking for long-running jobs.
- Configure logging to see why an adapter returned nothing.

See also: [03 Rate limits, retries and timeouts](03-rate-limiting.ipynb) and the
[per-source examples](../README.md).